In [ ]:
# ==========================================
# 1. Install Required Packages
# ==========================================
!pip install -q xee xarray netcdf4 geopandas geemap pyproj

import os
import glob
import ee
import xarray as xr
import xee
from xee import helpers
import geopandas as gpd
from google.colab import drive
import shapely
import shutil
import pyproj
import dask
import requests
import urllib3
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

# 1. Increase urllib3 connection pool size so connections aren't dropped
adapter = requests.adapters.HTTPAdapter(pool_connections=50, pool_maxsize=50)
session = requests.Session()
session.mount('https://', adapter)

# 2. Limit Dask parallel worker threads (prevents hammering GEE all at once)
dask.config.set(scheduler='threads', num_workers=7)
# Mount Google Drive to save .nc files directly
drive.mount('/content/drive')

# ==========================================
# 2. Authenticate & Initialize Earth Engine
# ==========================================
ee.Authenticate()

PROJECT_ID = "msugw-503806"
ee.Initialize(project=PROJECT_ID)

# ==========================================
# 3. Load & Reproject Shapefile ROI
# ==========================================
extract_path = '/content/ogallala_shp'

shp_files = glob.glob(os.path.join(extract_path, "**", "*.shp"), recursive=True)
if not shp_files:
    raise FileNotFoundError(f"No .shp file found inside '{extract_path}'. Please verify path.")

shp_path = shp_files[0]
print(f"Loading shapefile: {shp_path}")

gdf = gpd.read_file(shp_path)

# Ensure shapefile is in WGS 84 (EPSG:4326)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    print("Reprojecting shapefile to EPSG:4326...")
    gdf = gdf.to_crs(epsg=4326)

ogallala_geom_shapely = gdf.geometry.union_all()          # for fit_geometry
ogallala_geom = ee.Geometry(ogallala_geom_shapely.__geo_interface__)  # for EE-side use, if needed

# ==========================================
# 4. Prepare Google Drive Output Folder
# ==========================================
dataset_id = "USDA/NASS/CDL"
folder_name = dataset_id.split("/")[-1]  # Automatically sets folder to 'CDL'

drive_folder = f"/content/drive/MyDrive/MSUGWB/{folder_name}"
os.makedirs(drive_folder, exist_ok=True)

# ==========================================
# 5. Load Collection & Metadata Dynamically
# ==========================================
full_collection = ee.ImageCollection(dataset_id)

# Dynamically fetch ALL variables/bands present in the dataset (cropland, confidence, cultivated, etc.)
target_variables = full_collection.first().bandNames().getInfo()

# Determine available year range from the dataset metadata
start_year = ee.Date(full_collection.sort('system:time_start').first().get('system:time_start')).get('year').getInfo()
end_year = ee.Date(full_collection.sort('system:time_start', False).first().get('system:time_start')).get('year').getInfo()
years = list(range(start_year, end_year + 1))

print(f"Dataset: {dataset_id}")
print(f"Variables ({len(target_variables)} total): {target_variables}")
print(f"Processing Years: {start_year} to {end_year}")

# ==========================================
# 6. Native 30m Grid & CRS Determination (CONUS Albers / EPSG:5070)
# ==========================================
base_ic = full_collection.select(target_variables)
source_params = helpers.extract_grid_params(base_ic)
source_crs = source_params.get('crs', 'EPSG:5070')

use_native_crs = True
try:
    pyproj.CRS.from_user_input(source_crs)
    grid_crs = source_crs
    source_transform = source_params['crs_transform']
    # Preserve native 30m x 30m resolution (in meters)
    grid_scale = (abs(source_transform[0]), -abs(source_transform[4]))
    print(f"Using Native Projection: {grid_crs} | Grid Scale: {grid_scale}")
except Exception as e:
    print(f"\n[Warning] Native CRS '{source_crs}' could not be parsed by pyproj.")
    print("Defaulting to native USDA CDL projection (EPSG:5070) at 30m resolution...")
    grid_crs = 'EPSG:5070'
    grid_scale = (30.0, -30.0)

# ==========================================
# 7. Stream & Export 3D NetCDF (.nc) via XEE
# ==========================================
grid_params = helpers.fit_geometry(
    geometry=ogallala_geom_shapely,
    geometry_crs='EPSG:4326',
    grid_crs=grid_crs,
    grid_scale=grid_scale,
)

for year in years:
    output_path = os.path.join(drive_folder, f"{folder_name}_{year}_Ogallala.nc")
    if os.path.exists(output_path):
        print(f"[{year}] Already exists: {output_path} (Skipping)")
        continue

    print(f"\n[{year}] Opening Earth Engine collection via XEE...")
    year_col = (ee.ImageCollection(dataset_id)
                .filter(ee.Filter.calendarRange(year, year, 'year'))
                .select(target_variables)
                .sort('system:time_start'))

    # Check if data exists for this specific year
    if year_col.size().getInfo() == 0:
        print(f"[{year}] No images found for this year. Skipping.")
        continue

    ds = xr.open_dataset(
        year_col,
        engine='ee',
        chunks={'x': 512, 'y': 512},
        **grid_params,
    )

    print(f"[{year}] Streaming to NetCDF: {output_path}...")

    local_tmp = f"/content/{folder_name}_{year}_Ogallala.nc"
    ds.to_netcdf(local_tmp, engine='netcdf4')

    size = os.path.getsize(local_tmp)
    print(f"[{year}] Local file size: {size} bytes")

    if size < 1000:  # Sanity check
        raise RuntimeError(f"[{year}] Export produced a near-empty file ({size} bytes) — investigate before copying to Drive.")

    shutil.move(local_tmp, output_path)
    print(f"[{year}] Moved to Drive: {output_path}")
    print(f"[{year}] Successfully saved: {output_path}")